[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/03_reinforcement_active_learning.ipynb)

# 03 · Aspek *Reinforcement* — *Active Learning* yang Dikendalikan *Multi-Armed Bandit*

Label mahal di ketiga topik: uji lab kelarutan, telur yang harus dipecah, serta panel sensori dan
penyimpanan 48 jam. Setiap ronde, **agen** memilih satu *arm* (strategi kueri): `random`,
`uncertainty`, atau `diversity`. *Oracle* memberi label pada satu batch, model dilatih ulang, dan
**reward** = perbaikan relatif skor **validasi**. Data uji hanya dipakai untuk pelaporan.

- **ε-greedy + estimasi Monte Carlo:** Q(a) ← Q(a) + (r − Q(a)) / n(a)
- **UCB1:** pilih argmax Q(a) + c·√(2 ln N / n(a))

Gagasan ini mengikuti *Active Learning by Learning* (Hsu & Lin, AAAI 2015), dengan kebijakan bandit dari materi kuliah.

## 0. Instalasi & utilitas

In [ ]:
# Versi dikunci sama dengan uv.lock di repo supaya angka di Colab = angka di laporan
%pip install -q scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 optuna==5.0.0 \
    imbalanced-learn==0.14.2 rdkit==2026.9.1 shap==0.54.1

In [ ]:
import warnings, hashlib, json, time, urllib.request, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.base import clone
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              StackingClassifier, StackingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")
SEED = 42
RAW, RESULTS = Path("data/raw"), Path("results")
RAW.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(url, dest):
    import ssl, certifi                       # certifi: CA bundle yang sama di Colab dan macOS
    dest = Path(dest)
    if not dest.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "ets-ml2"})
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(req, context=ctx) as r, open(dest, "wb") as f:
            f.write(r.read())
    return dest

def mean_sd(v):
    v = np.asarray(v, float)
    return f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}"

## 1. Agen bandit & strategi kueri

In [ ]:
from sklearn.metrics import f1_score, mean_squared_error
ARMS = ["random", "uncertainty", "diversity"]

def q_random(ctx, k, rng): return rng.choice(ctx["pool"], size=k, replace=False)
def q_uncertainty(ctx, k, rng):
    u = ctx["uncertainty"](ctx["model"], ctx["X"][ctx["pool"]]); return ctx["pool"][np.argsort(-u)[:k]]
def q_diversity(ctx, k, rng):
    """k-center greedy: ambil titik pool yang paling jauh dari semua titik berlabel."""
    Z, pool = ctx["Z"], ctx["pool"]; d = ctx["min_dist"][pool].copy(); chosen = []
    for _ in range(k):
        j = int(np.argmax(d)); chosen.append(pool[j])
        d = np.minimum(d, np.linalg.norm(Z[pool] - Z[pool[j]], axis=1))
    return np.array(chosen)
QUERY = {"random": q_random, "uncertainty": q_uncertainty, "diversity": q_diversity}

def forest_std(model, X): return np.stack([t.predict(X) for t in model.estimators_]).std(axis=0)
def proba_entropy(model, X):
    p = np.clip(model.predict_proba(X), 1e-12, 1); return -(p * np.log(p)).sum(axis=1)

class Fixed:
    def __init__(self, arm): self.arm = arm
    def select(self, t, rng): return self.arm
    def update(self, arm, r): pass
class EpsilonGreedy:
    def __init__(self, arms, epsilon=0.2):
        self.arms, self.eps = arms, epsilon; self.n = {a: 0 for a in arms}; self.q = {a: 0.0 for a in arms}
    def select(self, t, rng):
        untried = [a for a in self.arms if self.n[a] == 0]
        if untried: return untried[0]
        if rng.random() < self.eps: return self.arms[rng.integers(len(self.arms))]
        return max(self.arms, key=self.q.get)
    def update(self, arm, r):
        self.n[arm] += 1; self.q[arm] += (r - self.q[arm]) / self.n[arm]
class UCB1(EpsilonGreedy):
    def __init__(self, arms, c=1.0): super().__init__(arms); self.c = c
    def select(self, t, rng):
        untried = [a for a in self.arms if self.n[a] == 0]
        if untried: return untried[0]
        N = sum(self.n.values())
        return max(self.arms, key=lambda a: self.q[a] + self.c * np.sqrt(2 * np.log(N) / self.n[a]))
def make_policy(name):
    return Fixed(name) if name in ARMS else EpsilonGreedy(ARMS, 0.2) if name == "bandit-eps" else UCB1(ARMS, 1.0)

def run_active_learning(model, Xp, yp, Xv, yv, Xt, yt, *, policy, score, uncertainty, n_init, batch,
                        rounds, seed, higher_is_better=True, stratify_init=False):
    rng = np.random.default_rng(seed); Xp = np.asarray(Xp); Z = StandardScaler().fit_transform(Xp); n = len(Xp)
    if stratify_init:
        lab = [rng.choice(np.flatnonzero(yp == c)) for c in np.unique(yp)]
        lab += list(rng.choice(np.setdiff1d(np.arange(n), lab), n_init - len(lab), replace=False)); lab = np.array(lab)
    else:
        lab = rng.choice(n, n_init, replace=False)
    mask = np.ones(n, bool); mask[lab] = False
    min_dist = np.full(n, np.inf)
    for i in lab: min_dist = np.minimum(min_dist, np.linalg.norm(Z - Z[i], axis=1))
    pol, sign = make_policy(policy), (1 if higher_is_better else -1)
    def fit_eval(idx):
        m = clone(model).fit(Xp[idx], yp[idx]); return m, score(yv, m.predict(Xv)), score(yt, m.predict(Xt))
    m, v, te_s = fit_eval(lab)
    log = [{"round": 0, "n_labelled": len(lab), "arm": "init", "val": v, "test": te_s, "reward": 0.0}]
    for t in range(1, rounds + 1):
        arm = pol.select(t, rng)
        ctx = {"pool": np.flatnonzero(mask), "X": Xp, "Z": Z, "model": m, "uncertainty": uncertainty, "min_dist": min_dist}
        new = QUERY[arm](ctx, min(batch, mask.sum()), rng); mask[new] = False
        for i in new: min_dist = np.minimum(min_dist, np.linalg.norm(Z - Z[i], axis=1))
        lab = np.concatenate([lab, new]); m, v_new, te_s = fit_eval(lab)
        r = sign * (v_new - v) / (abs(v) + 1e-9); pol.update(arm, r); v = v_new
        log.append({"round": t, "n_labelled": len(lab), "arm": arm, "val": v, "test": te_s, "reward": r})
    return log
POLICIES = ["random", "uncertainty", "diversity", "bandit-eps", "bandit-ucb"]

## 2. AL pada kelarutan (AqSolDB)
*Pool* = data latih scaffold, validasi = scaffold split 15% di dalam data latih, uji = scaffold uji.
Mulai dari 100 label, lalu +100 per ronde selama 20 ronde (total 2.100 label), dengan 5 seed.

In [ ]:
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")
parts = [pd.read_csv(fetch(f"https://huggingface.co/datasets/maomlab/AqSolDB/resolve/main/AqSolDB/{f}", RAW / f"aqsoldb_{f}"))
         for f in ["train.csv", "test.csv"]]
df = pd.concat(parts, ignore_index=True).drop_duplicates("InChIKey").rename(columns={"Y": "logS"}).reset_index(drop=True)
def scaffold_of(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return f"INVALID:{smi}"
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception: return f"INVALID:{smi}"
df["scaffold"] = df["SMILES"].map(scaffold_of)
empty = df["scaffold"] == ""; df.loc[empty, "scaffold"] = "ACYCLIC:" + df.loc[empty, "InChIKey"]
rows = []
for smi in df["SMILES"]:
    mol = Chem.MolFromSmiles(smi); rows.append([fn(mol) if mol is not None else np.nan for _, fn in Descriptors.descList])
X = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
X = X.loc[:, X.isna().mean() < 0.01].clip(-1e6, 1e6); X = X.loc[:, X.std() > 0].astype("float32").to_numpy()
y = df["logS"].to_numpy()
def scaffold_split(d, test_frac=0.2, seed=SEED):
    groups = defaultdict(list)
    for i, s in enumerate(d["scaffold"]): groups[s].append(i)
    keys = list(groups); np.random.default_rng(seed).shuffle(keys); keys.sort(key=lambda k: len(groups[k]), reverse=True)
    n_train, a, b = int(round((1 - test_frac) * len(d))), [], []
    for k in keys: (a if len(a) + len(groups[k]) <= n_train else b).extend(groups[k])
    return np.array(sorted(a)), np.array(sorted(b))
tr, te = scaffold_split(df)
ptr, pval = scaffold_split(df.iloc[tr].reset_index(drop=True), test_frac=0.15); pool, val = tr[ptr], tr[pval]
X = np.where(np.isnan(X), np.nanmedian(X[pool], axis=0), X)          # imputasi median dari pool saja
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=2, max_features=0.3, n_jobs=-1, random_state=SEED)
full = rmse(y[te], clone(rf).fit(X[pool], y[pool]).predict(X[te]))
print(f"pool={len(pool)} val={len(val)} test={len(te)}  RMSE jika SELURUH pool dilabeli = {full:.3f}")
logs = []
for seed in range(5):
    for pol in POLICIES:
        log = run_active_learning(rf, X[pool], y[pool], X[val], y[val], X[te], y[te], policy=pol, score=rmse,
                                  uncertainty=forest_std, n_init=100, batch=100, rounds=20, seed=seed,
                                  higher_is_better=False)
        logs += [dict(r, policy=pol, seed=seed) for r in log]
        print(f"seed={seed} {pol:12s} RMSE akhir={log[-1]['test']:.3f}")
al_sol = pd.DataFrame(logs); al_sol.to_csv(RESULTS / "al_solubility.csv", index=False)

In [ ]:
def plot_curves(al, ylabel, title, ref=None):
    c = al.groupby(["policy", "n_labelled"]).test.mean().unstack(0)
    ax = c.plot(figsize=(8, 4.5), marker="o", ms=3, title=title); ax.set_xlabel("jumlah label"); ax.set_ylabel(ylabel)
    if ref is not None: ax.axhline(ref, ls="--", c="k", lw=.8, label="seluruh pool")
    ax.legend(); plt.show(); return c
c = plot_curves(al_sol, "RMSE uji (scaffold)", "Kurva belajar AL - kelarutan", full)
arms = al_sol[al_sol.policy.str.startswith("bandit") & (al_sol.arm != "init")]
display(arms.groupby("policy").arm.value_counts(normalize=True).unstack().round(2))
target = c["random"].iloc[-1]
print("label yang dibutuhkan untuk menyamai RMSE akhir random:",
      {p: next((int(n) for n, s in c[p].items() if s <= target), None) for p in c})

## 3. AL pada *e-nose* (biner segar vs tidak segar)
Untuk setiap seri uji, satu seri lain menjadi validasi dan tiga sisanya menjadi *pool*.
Mulai dari 40 label, lalu +40 per ronde selama 15 ronde, dengan 3 seed.

In [ ]:
# Akuisisi: e-nose daging sapi (Wijaya dkk., 2018, Data in Brief 21) - Mendeley Data, CC BY 4.0
ENOSE_URL = ("https://data.mendeley.com/public-files/datasets/mwmhh766fc/files/"
             "2e65305f-5e53-4395-9b28-91dfc73ff490/file_downloaded")
ENOSE_SHA256 = "772bc2c75f2ac4b785538ea38311bdefaee65059b3391b080b4da5ea619963e3"
z = fetch(ENOSE_URL, RAW / "enose_beef.zip")
assert sha256(z) == ENOSE_SHA256, "checksum tidak cocok - file berubah/korup"
zipfile.ZipFile(z).extractall(RAW / "enose_beef")

CLASSES = ["excellent", "good", "acceptable", "spoiled"]      # ordinal: segar -> busuk
GAS = ["MQ135", "MQ136", "MQ2", "MQ3", "MQ4", "MQ5", "MQ6", "MQ8", "MQ9"]
PREDICTORS_RAW = GAS + ["Humidity"]
BASELINE_MIN, WINDOW = 10, 15

def load_enose():
    frames = []
    for f in sorted(glob.glob(str(RAW / "enose_beef" / "TS*.csv"))):
        d = pd.read_csv(f); d["series"] = Path(f).stem; frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    d["y"] = d["class"].map({c: i for i, c in enumerate(CLASSES)})
    return d

enose = load_enose()
print(enose.shape); enose.head()

In [ ]:
def engineer(g):
    """Fitur kausal satu seri: rasio terhadap baseline 10 menit pertama, rata-rata bergulir,
    dan kemiringan 15 menit. Semua hanya memakai data masa lalu (bisa dihitung real time)."""
    g = g.sort_values("minute")
    out = pd.DataFrame(index=g.index)
    for c in GAS:
        ratio = g[c] / g[c].iloc[:BASELINE_MIN].mean()
        out[f"{c}_ratio"] = ratio
        out[f"{c}_ratio_roll"] = ratio.rolling(WINDOW, min_periods=1).mean()
        out[f"{c}_slope"] = ratio.diff(WINDOW).fillna(0) / WINDOW
    out["Humidity"] = g["Humidity"]
    return out

def build_features(d, kind="engineered", with_temperature=False):
    X = d[PREDICTORS_RAW].copy() if kind == "raw" else \
        pd.concat([engineer(g) for _, g in d.groupby("series", sort=False)]).loc[d.index]
    if with_temperature:
        X["Temperature"] = d["Temperature"]
    return X

In [ ]:
Xe = build_features(enose).to_numpy(); ye = (enose["y"].to_numpy() >= 2).astype(int)
series = enose.series.to_numpy(); names = sorted(np.unique(series))
xgb = XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                    n_jobs=4, random_state=SEED, verbosity=0)
f1 = lambda a, b: float(f1_score(a, b, average="macro"))
logs = []
for i, test_s in enumerate(names):
    val_s = names[(i + 1) % 5]
    te_i, val_i = np.flatnonzero(series == test_s), np.flatnonzero(series == val_s)
    pool_i = np.flatnonzero((series != test_s) & (series != val_s))
    full = f1(ye[te_i], clone(xgb).fit(Xe[pool_i], ye[pool_i]).predict(Xe[te_i]))
    for seed in range(3):
        for pol in POLICIES:
            log = run_active_learning(xgb, Xe[pool_i], ye[pool_i], Xe[val_i], ye[val_i], Xe[te_i], ye[te_i],
                                      policy=pol, score=f1, uncertainty=proba_entropy, n_init=40, batch=40,
                                      rounds=15, seed=seed, stratify_init=True)
            logs += [dict(r, policy=pol, seed=seed, test_series=test_s, full_pool_test=full) for r in log]
    print(test_s, "selesai; F1 seluruh pool =", round(full, 3))
al_en = pd.DataFrame(logs); al_en.to_csv(RESULTS / "al_enose.csv", index=False)
plot_curves(al_en, "macro-F1 uji (seri baru)", "Kurva belajar AL - e-nose biner", al_en.full_pool_test.mean());

**Bacaan:** pada *e-nose*, 40 label sudah memberi F1 ≈ 0,83 dan kurvanya hampir datar. Kendalanya
**bukan jumlah label, melainkan pergeseran domain antar seri** (lihat notebook 01 §9). Strategi pintar
yang memilih titik "aneh" dari seri yang sama tidak membantu, dan *random* justru paling aman. Bandit
dapat membatasi kerugian dengan memilih arm yang terbukti baik pada validasi, tetapi tidak dapat
menciptakan informasi yang tidak ada di dalam *pool*.